In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

print(QDRANT_URL[:50])
print("API Key Loaded")

https://fa199dbc-d1ea-4738-ad4a-4232f2a66bc9.eu-ce
API Key Loaded


In [2]:
import pandas as pd

df = pd.read_csv("processed_stackoverflow.csv")

print(df.shape)

(50000, 8)


In [3]:
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():

    content = f"""
Title: {row['Title']}

Question:
{row['QuestionBody']}

Tags:
{row['Tags']}

Answer:
{row['AnswerBody']}
"""

    documents.append(
        Document(
            page_content=content,
            metadata={
                "question_id": int(row["Id"]),
                "tags": str(row["Tags"])
            }
        )
    )

print(len(documents))

50000


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model ready")

c:\Users\bilal\Desktop\Crosstab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 649.69it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model ready


In [5]:
from qdrant_client import QdrantClient

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

print(client.get_collections())

collections=[]


In [27]:
from qdrant_client.models import Distance, VectorParams

try:
    client.create_collection(
        collection_name="python_qa",
        vectors_config=VectorParams(
            size=384,
            distance=Distance.COSINE
        )
    )

    print("Collection Created!")

except Exception:
    print("Collection already exists")

Collection already exists


In [ ]:
from langchain_qdrant import QdrantVectorStore

qdrant_store = QdrantVectorStore(
    client=client,
    collection_name="python_qa",
    embedding=embedding_model
)

print("Qdrant Store Ready")

In [23]:
batch_size = 1000

for i in range(0, len(documents), batch_size):

    batch = documents[i:i + batch_size]

    print(f"Uploading {i} to {i + len(batch)}")

    qdrant_store.add_documents(batch)

    print("Done")

Uploading 0 to 1000
Done
Uploading 1000 to 2000
Done
Uploading 2000 to 3000
Done
Uploading 3000 to 4000
Done
Uploading 4000 to 5000
Done
Uploading 5000 to 6000
Done
Uploading 6000 to 7000
Done
Uploading 7000 to 8000
Done
Uploading 8000 to 9000
Done
Uploading 9000 to 10000
Done
Uploading 10000 to 11000
Done
Uploading 11000 to 12000
Done
Uploading 12000 to 13000
Done
Uploading 13000 to 14000
Done
Uploading 14000 to 15000
Done
Uploading 15000 to 16000
Done
Uploading 16000 to 17000
Done
Uploading 17000 to 18000
Done
Uploading 18000 to 19000
Done
Uploading 19000 to 20000
Done
Uploading 20000 to 21000
Done
Uploading 21000 to 22000
Done
Uploading 22000 to 23000
Done
Uploading 23000 to 24000
Done
Uploading 24000 to 25000
Done
Uploading 25000 to 26000
Done
Uploading 26000 to 27000
Done
Uploading 27000 to 28000
Done
Uploading 28000 to 29000
Done
Uploading 29000 to 30000
Done
Uploading 30000 to 31000
Done
Uploading 31000 to 32000
Done
Uploading 32000 to 33000
Done
Uploading 33000 to 34000
Done
Up

In [25]:
print(
    "Final Point Count:",
    client.get_collection("python_qa").points_count
)

Final Point Count: 50000


In [26]:
retriever = qdrant_store.as_retriever(
    search_kwargs={"k": 3}
)

results = retriever.invoke(
    "How do Python generators work?"
)

print("Retrieved:", len(results), "documents")

for doc in results:
    print(doc.metadata)

Retrieved: 3 documents
{'question_id': 1756096, 'tags': 'python, generator', '_id': '3e459cfa-9867-4952-a625-8cf14c83bd46', '_collection_name': 'python_qa'}
{'question_id': 1995418, 'tags': 'python, yield, generator', '_id': '9a8207a0-b0c6-4223-9094-b5b9918f6de6', '_collection_name': 'python_qa'}
{'question_id': 29570348, 'tags': 'python, generator', '_id': 'b5d6bfac-641b-4d82-94fa-589b5d5ef931', '_collection_name': 'python_qa'}
